# Ladder Basics: One Prompt at a Time

This notebook looks at **one prompt at a time**, with no aggregates, to show exactly what the model sees and what the score is.

1. Load the model.
2. Choose one stimulus and one task prompt, and show the exact text the model receives.
3. Score it: the log-probability of every answer token, each answer's total log-probability, and the log odds between the two classes.

To explore, change the choices at the top of cell 2 and re-run cells 2 and 3.

## 1. Load the Model
This uses the same loader as the Stage A scorer (`src.interp_score.load_model`), at the revision recorded in the Stage A `run.json`. The scores therefore match `runs/llama3.2-3b/stage_A/scores.jsonl` exactly.

In [1]:
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("HF_HOME", "/fs/clip-scratch/rlab_models")
REPO = Path("/fs/nexus-projects/rlab/pramu/geovariance_of_llm_decisions")
sys.path.insert(0, str(REPO))

import pandas as pd
import torch

from src import interp_score as S

MODEL_KEY = "llama3.2-3b"
DEVICE = "cuda"

spec = S.MODELS[MODEL_KEY]
revision = json.loads((REPO / "interpretability/runs" / MODEL_KEY / "stage_A/run.json").read_text())["resolved_revision"]
tokenizer, model, revision = S.load_model(spec, revision, DEVICE)
print(f"Loaded {spec['hf_id']} ({spec['dtype']}) at {revision[:10]} on {torch.cuda.get_device_name()}")

/fs/clip-projects/rlab/pramu/miniconda3/envs/interp-pilot/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 254/254 [00:00<00:00, 33541.75it/s]

Loaded meta-llama/Llama-3.2-3B-Instruct (bfloat16) at 0cb88a4f76 on NVIDIA RTX A4000


## 2. Choose One Prompt and See What the Model Receives
A prompt is one **stimulus** (the event description, sent as the user message) combined with one **task prompt** (the system message, set by the task version and the class order). Both come from the frozen Stage A snapshot in `interpretability/data/stimuli/`.

The choices at the top of the cell:
- `BOUNDARY`: `"PR"` (Protest vs Riot) or `"BV"` (Battle vs Violence against civilians)
- `BASE`: `"pr_01"`–`"pr_24"` or `"bv_01"`–`"bv_24"`
- `LEVEL`: 1–5, where 1 is the lower-class anchor and 5 is the higher-class anchor
- `WORDING`: 1 or 2
- `VERSION`: `"defined"` or `"undefined"`
- `ORDER`: `"A"` or `"B"`, which class is listed first
- `USER_TEXT`: leave as `None` to use the snapshot stimulus, or set it to your own event description to score that instead. The task prompt still comes from `BOUNDARY`, `VERSION` and `ORDER`.

The cell prints the system message, the user message, and the **full rendered chat template**, which is the exact string that gets tokenized. The model's answer begins right after the last line, the assistant header.

In [2]:
BOUNDARY = "PR"
BASE = "pr_01"
LEVEL = 3
WORDING = 1
VERSION = "defined"
ORDER = "A"
USER_TEXT = None  # e.g. "On 12 June 2025, students gathered outside a government office to demonstrate against rising fuel prices. ..."

SNAP = REPO / "interpretability/data/stimuli"
stimuli = pd.read_csv(SNAP / "stimuli.csv").set_index("stimulus_id")
tasks = pd.read_csv(SNAP / "system_prompts.csv").set_index("task_id")

stimulus_id = f"{BOUNDARY}:{BASE}:L{LEVEL}:W{WORDING}:none"
task_id = f"{BOUNDARY}:{VERSION}:{ORDER}"
stimulus = stimuli.loc[stimulus_id]
task = tasks.loc[task_id]
system_prompt = task.system_prompt
if USER_TEXT is None:
    user_text = stimulus.text
    SOURCE = f"stimulus {stimulus_id} ({stimulus.level_role}); evidence: {stimulus.evidence}"
else:
    user_text = " ".join(USER_TEXT.split())
    SOURCE = "custom USER_TEXT (not from the snapshot)"
LOWER, HIGHER = task.lower_class, task.higher_class

prompt_text, prompt_ids = S.render_prompt(tokenizer, spec, system_prompt, user_text)

print(f"task {task_id}\n{SOURCE}\n")
print("=" * 30, "SYSTEM MESSAGE", "=" * 30)
print(system_prompt)
print("\n" + "=" * 30, "USER MESSAGE", "=" * 30)
print(user_text)
print("\n" + "=" * 30, f"FULL RENDERED PROMPT ({len(prompt_ids)} tokens)", "=" * 30)
print(prompt_text, end="")
print("<<< answer starts here")

task PR:defined:A
stimulus PR:pr_01:L3:W1:none (ambiguous); evidence: A nearby car was damaged; it is unclear whether participants or bystanders were responsible.

============================== SYSTEM MESSAGE ==============================
You code demonstration events for a conflict-event dataset. Read the event description and decide whether the event is a Protest or a Riot.

Protest: a public demonstration in which participants remain nonviolent, including demonstrations that face intervention or violence from others.

Riot: a demonstration in which participants engage in violence or destruction, such as attacking people or damaging property.

Answer with one word: Protest or Riot.

============================== USER MESSAGE ==============================
On 12 June 2025, students gathered outside a government office to protest rising fuel prices. A nearby car was damaged; it is unclear whether participants or bystanders were responsible. The gathering ended by evening.

=========

## 3. Log-Probabilities and Log Odds for This Prompt
Each answer can span several tokens; Riot, for example, is `R` + `iot`. The cell scores each answer by **teacher forcing**:
1. Append the answer's tokens to the prompt and run one forward pass.
2. For each answer token, read its log-probability from the logit row **before** it. Row *i* predicts token *i + 1*.
3. Sum these per-token values to get the answer's log-probability.

The cell does this explicitly and checks the result against `S.score_prompt`, the function Stage A used. It then shows:
- **Per-token table:** each answer token with its log-probability and probability. Usually the first token carries the decision, and later tokens are close to 0.
- **Per-answer table:** log P(answer), P(answer), and P renormalized over the two answers.
- **Log odds** d = log P(higher) − log P(lower): d > 0 favours the higher-evidence class (Riot or Battle), and d = 0 means a coin flip.
- **Top-10 next tokens at the answer position:** what the model wants to say first, which shows how much probability falls outside the two answers.

In [3]:
import math

answers = S.resolve_answers(tokenizer, prompt_text, prompt_ids, [LOWER, HIGHER])  # {class: [token ids]}

token_rows, totals = [], {}
for name, answer_ids in answers.items():
    ids = list(prompt_ids) + list(answer_ids[:-1])  # the last answer token is predicted, never fed in
    with torch.inference_mode():
        logits = model(input_ids=torch.tensor([ids], device=DEVICE)).logits[0]
    log_probs = torch.log_softmax(logits.float(), dim=-1)
    total = 0.0
    for k, tok in enumerate(answer_ids):
        row = len(prompt_ids) - 1 + k  # the row that predicts answer token k
        lp = float(log_probs[row, tok])
        total += lp
        token_rows.append(dict(answer=name, position=k, token=repr(tokenizer.decode([tok])), token_id=tok, logprob=lp, prob=math.exp(lp)))
    totals[name] = total
    if name == LOWER:  # the first answer position is the same in both passes
        first_row = log_probs[len(prompt_ids) - 1]

# Cross-check against the scorer used for Stage A.
ref = S.score_prompt(model, tokenizer, spec, system_prompt, user_text, [LOWER, HIGHER], DEVICE)
assert all(abs(ref["answers"][n]["logprob"] - totals[n]) < 1e-4 for n in totals), "mismatch with S.score_prompt"

print(f"{task_id}  |  {SOURCE}\n")
print("Per-token log-probabilities")
display(pd.DataFrame(token_rows).style.format({"logprob": "{:.4f}", "prob": "{:.4g}"}).hide(axis="index"))

mass = sum(math.exp(v) for v in totals.values())
per_answer = pd.DataFrame(
    [dict(answer=n, logprob=v, prob=math.exp(v), prob_renormalized=math.exp(v) / mass) for n, v in totals.items()]
)
print("Per-answer totals")
display(per_answer.style.format({"logprob": "{:.4f}", "prob": "{:.4g}", "prob_renormalized": "{:.4f}"}).hide(axis="index"))

d = totals[HIGHER] - totals[LOWER]
print(f"log odds  d = log P({HIGHER}) - log P({LOWER}) = {totals[HIGHER]:.3f} - ({totals[LOWER]:.3f}) = {d:+.3f}")
print(f"          -> P({HIGHER} | one of the two) = sigmoid(d) = {1 / (1 + math.exp(-d)):.4f}")
print(f"answer mass P({LOWER}) + P({HIGHER}) = {mass:.4f}\n")

values, indices = torch.topk(first_row, 10)
print("Top-10 next tokens at the answer position")
display(
    pd.DataFrame(
        [dict(rank=i + 1, token=repr(tokenizer.decode([int(t)])), token_id=int(t), logprob=float(v), prob=float(v.exp())) for i, (v, t) in enumerate(zip(values, indices))]
    ).style.format({"logprob": "{:.3f}", "prob": "{:.4g}"}).hide(axis="index")
)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


PR:defined:A  |  stimulus PR:pr_01:L3:W1:none (ambiguous); evidence: A nearby car was damaged; it is unclear whether participants or bystanders were responsible.

Per-token log-probabilities


answer,position,token,token_id,logprob,prob
Protest,0,'Pro',1360,-0.0092,0.9909
Protest,1,'test',1985,-0.0000,1
Riot,0,'R',49,-4.7592,0.008573
Riot,1,'iot',11345,-0.0002,0.9998


Per-answer totals


answer,logprob,prob,prob_renormalized
Protest,-0.0092,0.9908,0.9914
Riot,-4.7594,0.008571,0.0086


log odds  d = log P(Riot) - log P(Protest) = -4.759 - (-0.009) = -4.750
          -> P(Riot | one of the two) = sigmoid(d) = 0.0086
answer mass P(Protest) + P(Riot) = 0.9994

Top-10 next tokens at the answer position


rank,token,token_id,logprob,prob
1,'Pro',1360,-0.009,0.9909
2,'R',49,-4.759,0.008573
3,'Prosec',86754,-9.259,9.523e-05
4,' Protest',37759,-9.509,7.417e-05
5,'-Pro',58186,-9.634,6.545e-05
6,'Pros',46015,-10.009,4.499e-05
7,'Prot',12705,-10.384,3.092e-05
8,'Probe',82526,-11.509,1.004e-05
9,'Prov',81361,-11.822,7.344e-06
10,'Protocol',21346,-11.884,6.899e-06


## 4. Audit: Can We Write Genuinely Ambiguous Event Descriptions?
This section reads v1 as a linguist would: what does each sentence **assert**, what does it **presuppose**, and which label do those meanings settle? It sets out six hypotheses about why v1 isn't ambiguous, then tests each one with **minimal pairs**, single descriptions that differ in one feature. Every description is scored on all four task prompts, with **no averaging over bases**: each row is one description on one base (pr_01, bv_01).

| # | Hypothesis | Where it shows up in v1 |
| --- | --- | --- |
| H-a | **Label leak.** The frame names a class, or uses a definition's own words. | PR: "to **protest**" in every stimulus. BV: "the people **attacked**" echoes the definition "deliberately **attacks** unarmed civilians." |
| H-b | **The frame presupposes the event structure.** "Members of an armed group *opened fire on* a group of people" makes the group the agent and the people the patients, a one-sided attack. That *is* Violence against civilians. A definite description like "the people attacked" carries the presupposition, and it survives under "it is unclear whether …" (presupposition projection). So the hedge only questions whether they were armed, never who attacked whom. | BV frame and all BV evidence |
| H-c | **Uncertainty about a fact is not uncertainty about the label.** Each pair has one *default* class (Protest; Violence against civilians) and one *marked* class that needs positive evidence (Riot; Battle). "Unclear whether X" or "unconfirmed reports of X" means the evidence for the marked class is missing, so a rule-follower picks the default. The PR definition makes this explicit: Protest includes demonstrations facing "violence from others", so violence by an unknown party → Protest. | v1 L2–L4 are all hedges about a fact |
| H-d | **Vague acts may work better than hedges.** A *borderline* act whose facts are fully known (eggs, graffiti, toppling a barrier, an armed self-defence group) leaves the label itself open. Human annotators really do disagree on such cases. | not tested in v1 |
| H-e | **Clause order and source credibility.** In "A said X, **but** B said Y", the clause after *but* tends to win, and an interested source (organizers, attackers) carries less weight than a neutral one (local officials). | v1 PR L2 and BV L2 put the claim favouring the lower class last, or give it the more credible source |
| H-f | **Closing sentences set the tone.** "The gathering ended by evening" signals an uneventful end. "Two **people** were killed" describes the dead as people, not fighters. | both frames |

### Scoring Helper
`score4(boundary, text)` scores one description on all four task prompts with the same scorer as cell 3. It returns one row with d for each prompt and the order average for each version. In the printed tables, `p_def` and `p_und` are sigmoid(order-averaged d): P(higher class) for that version.

In [4]:
import math


def score4(boundary, text):
    row = {}
    for version in ("defined", "undefined"):
        for order in ("A", "B"):
            t = tasks.loc[f"{boundary}:{version}:{order}"]
            r = S.score_prompt(model, tokenizer, spec, t.system_prompt, " ".join(text.split()), [t.lower_class, t.higher_class], DEVICE)
            a = r["answers"]
            row[f"{version[:3]}_{order}"] = a[t.higher_class]["logprob"] - a[t.lower_class]["logprob"]
        row[f"{version[:3]}_avg"] = (row[f"{version[:3]}_A"] + row[f"{version[:3]}_B"]) / 2
    return row


def audit(boundary, items):
    # items: list of (id, hypothesis, text); prints the text, returns one row per description
    rows = []
    for item_id, hyp, text in items:
        rows.append(dict(id=item_id, tests=hyp, **score4(boundary, text)))
        print(f"{item_id:>4} [{hyp}] {' '.join(text.split())}")
    df = pd.DataFrame(rows).set_index("id")
    df["p_def"] = 1 / (1 + (-df.def_avg).map(math.exp))
    df["p_und"] = 1 / (1 + (-df.und_avg).map(math.exp))
    return df


def show(df):
    cols = ["def_A", "def_B", "def_avg", "und_A", "und_B", "und_avg"]
    return (
        df.style.format({c: "{:+.2f}" for c in cols} | {"p_def": "{:.3f}", "p_und": "{:.3f}"})
        .background_gradient(cmap="RdBu_r", vmin=-8, vmax=8, subset=cols)
        .background_gradient(cmap="RdBu_r", vmin=0, vmax=1, subset=["p_def", "p_und"])
    )

### PR Minimal Pairs (Base pr_01: Students, Fuel Prices, Government Office)
d = log P(Riot) − log P(Protest). Blue means Protest, red means Riot, white means ambiguous.
- `P0`–`P2` take v1 L3 and remove the label leak (H-a), then the closing sentence (H-f).
- `P3` is a hedged fact: is uncertainty about who acted enough (H-c)?
- `P4`–`P8` are borderline acts with the facts fully known (H-d).
- `P9`–`P12` are conflicting reports: clause order and source credibility (H-e).

From `P2` on, the frame is "to demonstrate against" with no closing sentence.

In [5]:
F = "On 12 June 2025, students gathered outside a government office to demonstrate against rising fuel prices."
V1_L3 = "A nearby car was damaged; it is unclear whether participants or bystanders were responsible."
pr_items = [
    ("P0", "v1 as-is", f"On 12 June 2025, students gathered outside a government office to protest rising fuel prices. {V1_L3} The gathering ended by evening."),
    ("P1", "H-a", f"{F} {V1_L3} The gathering ended by evening."),
    ("P2", "H-a+f", f"{F} {V1_L3}"),
    ("P3", "H-c", f"{F} It is unclear whether participants threw stones."),
    ("P4", "H-d", f"{F} Some participants threw eggs at the building."),
    ("P5", "H-d", f"{F} Some participants spray-painted slogans on the walls of the building."),
    ("P6", "H-d", f"{F} Some participants pushed against a metal barrier until it fell over."),
    ("P7", "H-d", f"{F} Some participants blocked the road with burning tires."),
    ("P8", "H-d", f"{F} A handful of participants threw stones while the rest of the crowd stayed calm."),
    ("P9", "H-e", f"{F} Some witnesses said participants threw stones; others said no stones were thrown."),
    ("P10", "H-e", f"{F} Some witnesses said no stones were thrown; others said participants threw stones."),
    ("P11", "H-e", f"{F} Organizers said no stones were thrown, but local officials said participants threw stones."),
    ("P12", "H-e", f"{F} Local officials said participants threw stones, but organizers said no stones were thrown."),
]
pr_audit = audit("PR", pr_items)
show(pr_audit)

  P0 [v1 as-is] On 12 June 2025, students gathered outside a government office to protest rising fuel prices. A nearby car was damaged; it is unclear whether participants or bystanders were responsible. The gathering ended by evening.


  P1 [H-a] On 12 June 2025, students gathered outside a government office to demonstrate against rising fuel prices. A nearby car was damaged; it is unclear whether participants or bystanders were responsible. The gathering ended by evening.


  P2 [H-a+f] On 12 June 2025, students gathered outside a government office to demonstrate against rising fuel prices. A nearby car was damaged; it is unclear whether participants or bystanders were responsible.


  P3 [H-c] On 12 June 2025, students gathered outside a government office to demonstrate against rising fuel prices. It is unclear whether participants threw stones.


  P4 [H-d] On 12 June 2025, students gathered outside a government office to demonstrate against rising fuel prices. Some participants threw eggs at the building.


  P5 [H-d] On 12 June 2025, students gathered outside a government office to demonstrate against rising fuel prices. Some participants spray-painted slogans on the walls of the building.


... (4 outputs left out)

 P10 [H-e] On 12 June 2025, students gathered outside a government office to demonstrate against rising fuel prices. Some witnesses said no stones were thrown; others said participants threw stones.


 P11 [H-e] On 12 June 2025, students gathered outside a government office to demonstrate against rising fuel prices. Organizers said no stones were thrown, but local officials said participants threw stones.


 P12 [H-e] On 12 June 2025, students gathered outside a government office to demonstrate against rising fuel prices. Local officials said participants threw stones, but organizers said no stones were thrown.


,tests,def_A,def_B,def_avg,und_A,und_B,und_avg,p_def,p_und
id,,,,,,,,,
P0,v1 as-is,-4.75,-10.00,-7.38,-10.25,-8.25,-9.25,0.001,0.000
P1,H-a,-3.50,-5.38,-4.44,-10.00,-6.63,-8.31,0.012,0.000
P2,H-a+f,-1.75,-4.75,-3.25,-8.63,-6.75,-7.69,0.037,0.000
P3,H-c,-8.38,-8.25,-8.31,-11.75,-8.38,-10.06,0.000,0.000
P4,H-d,+8.00,+1.75,+4.87,-2.13,-2.00,-2.06,0.992,0.113
P5,H-d,+0.12,-4.63,-2.25,-9.50,-5.13,-7.31,0.095,0.001
P6,H-d,+1.50,-2.50,-0.50,-5.00,-3.63,-4.31,0.378,0.013
P7,H-d,+10.37,+2.62,+6.50,-1.38,-2.25,-1.81,0.998,0.140
P8,H-d,-1.13,-4.63,-2.88,-11.00,-6.50,-8.75,0.053,0.000


### BV Minimal Pairs (Base bv_01: Village, Early Morning)
d = log P(Battle) − log P(Violence against civilians). Blue means Violence against civilians, red means Battle.
- `B0`–`B3` take v1 L3, drop the casualty line (H-f), then replace the one-sided "opened fire on" frame and "the people attacked" with a neutral event structure (H-a, H-b).
- `B4` is a hedged fact under the neutral frame (H-c).
- `B5`–`B9` are borderline capability or participation with the facts fully known (H-d).
- `B10`–`B11` are conflicting reports in both clause orders (H-e).

From `B3` on, the frame is the neutral "a shooting involving …", with no casualty sentence.

In [6]:
F1 = "On 12 June 2025, members of an armed group opened fire on a group of people in a village in the early morning."
FN = "On 12 June 2025, a shooting involving members of an armed group and a group of local men took place in a village in the early morning."
bv_items = [
    ("B0", "v1 as-is", f"{F1} It is unclear whether the people attacked were armed. Two people were killed."),
    ("B1", "H-f", f"{F1} It is unclear whether the people attacked were armed."),
    ("B2", "H-a", f"{F1} It is unclear whether those people were armed."),
    ("B3", "H-b", f"{FN} It is unclear whether the local men were armed."),
    ("B4", "H-c", f"{FN} It is unclear whether the local men fired any shots."),
    ("B5", "H-d", f"{FN} The local men belonged to a village self-defense group and carried rifles."),
    ("B6", "H-d", f"{FN} The local men carried hunting rifles but did not fire them."),
    ("B7", "H-d", f"{FN} One of the local men fired a hunting rifle into the air."),
    ("B8", "H-d", f"{FN} One of the local men fired back with a hunting rifle before the group fled."),
    ("B9", "H-d", f"{FN} The local men were guarding the village at night and some of them were armed."),
    ("B10", "H-e", f"{FN} Residents said the local men were unarmed farmers; the armed group said they were fighters who shot first."),
    ("B11", "H-e", f"{FN} The armed group said the local men were fighters who shot first; residents said they were unarmed farmers."),
]
bv_audit = audit("BV", bv_items)
show(bv_audit)

  B0 [v1 as-is] On 12 June 2025, members of an armed group opened fire on a group of people in a village in the early morning. It is unclear whether the people attacked were armed. Two people were killed.


  B1 [H-f] On 12 June 2025, members of an armed group opened fire on a group of people in a village in the early morning. It is unclear whether the people attacked were armed.


  B2 [H-a] On 12 June 2025, members of an armed group opened fire on a group of people in a village in the early morning. It is unclear whether those people were armed.


  B3 [H-b] On 12 June 2025, a shooting involving members of an armed group and a group of local men took place in a village in the early morning. It is unclear whether the local men were armed.


  B4 [H-c] On 12 June 2025, a shooting involving members of an armed group and a group of local men took place in a village in the early morning. It is unclear whether the local men fired any shots.


  B5 [H-d] On 12 June 2025, a shooting involving members of an armed group and a group of local men took place in a village in the early morning. The local men belonged to a village self-defense group and carried rifles.


... (3 outputs left out)

  B9 [H-d] On 12 June 2025, a shooting involving members of an armed group and a group of local men took place in a village in the early morning. The local men were guarding the village at night and some of them were armed.


 B10 [H-e] On 12 June 2025, a shooting involving members of an armed group and a group of local men took place in a village in the early morning. Residents said the local men were unarmed farmers; the armed group said they were fighters who shot first.


 B11 [H-e] On 12 June 2025, a shooting involving members of an armed group and a group of local men took place in a village in the early morning. The armed group said the local men were fighters who shot first; residents said they were unarmed farmers.


,tests,def_A,def_B,def_avg,und_A,und_B,und_avg,p_def,p_und
id,,,,,,,,,
B0,v1 as-is,-9.62,-11.62,-10.62,-6.37,-15.94,-11.16,0.000,0.000
B1,H-f,-7.37,-6.00,-6.69,-5.37,-14.87,-10.12,0.001,0.000
B2,H-a,-9.25,-11.00,-10.12,-7.00,-15.75,-11.37,0.000,0.000
B3,H-b,-4.62,-6.00,-5.31,-4.50,-12.31,-8.41,0.005,0.000
B4,H-c,-3.50,-5.25,-4.37,-5.37,-10.37,-7.87,0.012,0.000
B5,H-d,+1.63,-1.00,+0.31,+2.25,-5.00,-1.37,0.578,0.202
B6,H-d,-4.25,-2.87,-3.56,-6.37,-8.50,-7.44,0.028,0.001
B7,H-d,-3.75,+1.50,-1.12,-7.87,-5.87,-6.87,0.245,0.001
B8,H-d,-2.62,+1.25,-0.69,-4.25,-7.62,-5.94,0.335,0.003



### Audit Read-Out (Llama 3.2 bf16; One Base per Boundary; Run 2026-11-05)
Written from the two tables above. Each row is a single prompt, and different bases moved d by about 1–2 nats in earlier runs, so treat differences under about 2 nats as noise.

**1. Uncertainty about a fact never makes the label ambiguous (H-c confirmed).** "It is unclear whether participants threw stones" (P3, d ≈ −8 to −10) and "unclear whether the local men fired any shots" (B4, ≈ −4 to −10) both fall firmly on the default class. So do conflicting reports (P9–P12, B10–B11). The model behaves as if one class needs positive evidence: Riot and Battle are the marked classes, Protest and Violence against civilians the defaults, and missing evidence means the default. **v1's L2–L4 are all hedges, which is why they never became ambiguous.**

**2. Borderline acts with fully known facts do produce ambiguity (H-d confirmed, at least under the defined task).**
- PR: "pushed against a metal barrier until it fell over" (P6) gives p = 0.38 and graffiti (P5) gives 0.10, while eggs (P4) and burning tires (P7) are already clear Riots, p ≈ 0.99.
- BV: "belonged to a village self-defense group and carried rifles" (B5) gives p = 0.58 defined and 0.20 undefined, the only item ambiguous in both versions. A communal self-defence group is a classic borderline case in real conflict coding. "Fired back … before the group fled" (B8) gives 0.34. "Carried hunting rifles but did not fire them" (B6) gives 0.03: the model's Battle needs actual *exchange of fire*, not just the *capability* the definition mentions.

**3. The leak and frame effects are real but only part of the story.**
- PR: removing "to protest" raises d by about 3 nats under the defined task (P0 → P1). Dropping "ended by evening" adds about 1 more (P2).
- BV: the neutral frame ("a shooting involving …") instead of "opened fire on" adds 3–5 nats (B2 → B3). Changing "the people attacked" to "those people" changes nothing (B1 → B2), so the frame verb's one-sided structure matters, not the word *attacked*. **Confound:** B3 also changes "people" to "local men", which may itself suggest combatants. A follow-up minimal pair should separate the two.

**4. The defined and undefined tasks place the boundary in different spots.** Without definitions, the model's *riot* is the everyday English concept, which implies large-scale disorder. Even eggs and burning tires stay Protest (p ≤ 0.14), while ACLED's definition counts any participant violence or destruction. This is a difference in the concept itself, not noise. A single ladder can therefore be ambiguous for both versions only if it spans both boundaries.

**5. Position bias is largest exactly where the content is borderline.** P4: d_A = +8.0, d_B = +1.75. B7: d_A = −3.75, d_B = +1.5. When the content stops deciding, the order of the class names takes over. Averaging the two orders is essential, and the order effect should be reported per level.

**What this means for constructing the ladder.** Build levels 2–4 from **borderline facts stated as certain** (vagueness), not from **hedged facts** (epistemic uncertainty). This reverses PLAN.md's rule that steps add evidence about "who acted or how certain the report is". The rule was meant to stop the definitions from settling the middle levels, but P5 and P6 show that a borderline act is *not* settled by the definitions either. Next: check that the borderline items (P5, P6, B5, B7, B8) keep their position across all 24 bases and both models before building them into a ladder.
